# Runbook 2.4.a: Access logs and monitoring  (MODEL SOLUTION, instructor copy, do not distribute)

**Skill:** Detect Attacks · **Maps to:** 2.4.A · **Language:** Python 3 (pytest)

**Steps:** 1 Brief · 2 Plan · 3 Build · 4 Test · 5 Try it on the evidence · 6 Wrap up

## 1. Brief

**The situation.** You are building the detection engine for a physical access control monitoring system at Crestfield Analytics. Given a list of timestamped access log entries, identify every badge ID that triggers at least one of three alert rules:

- **Rule 0, Repeated denial burst:** 5 or more ACCESS DENIED events for the same badge within any 60-second sliding window.
- **Rule 1, After-hours valid access to a restricted door:** any GRANTED entry at a door whose name starts with `"restricted-"` between 22:00 and 05:59 (inclusive, using the entry's local time).
- **Rule 2, Location inconsistency:** the same badge granted at two different doors within a gap of 10 minutes (600 seconds) where those two doors are listed as geographically separated in a provided `far_pairs` set of frozensets.

**Lesson objective.** Interpret physical access log entries to identify anomalous patterns, after-hours entries, repeated failures, and tailgating gaps, and explain what each pattern signals to a security analyst.

### What comes in
- `entries`: `list[dict]`, each dict has:
  - `"ts"`: `int`, Unix timestamp (seconds)
  - `"badge"`: `str`, badge identifier e.g. `"B-0101"`
  - `"door"`: `str`, door identifier e.g. `"restricted-server-room-A"`
  - `"outcome"`: `str`, `"GRANTED"` or `"DENIED"`
- `far_pairs`: `set[frozenset]`, each element is a frozenset of two door names that are geographically far apart.

### What must come out
A **sorted list of unique badge ID strings** that triggered one or more rules. Empty list if no alerts.

- [ ] Read the situation and say in one sentence what the tool must decide
- [ ] Find the input and the output above

## 2. Plan

Work it out on paper before you write code.

### Worked examples
```python
entries = [
    {"ts": 1710230400, "badge": "B-0101", "door": "lobby", "outcome": "DENIED"},
    {"ts": 1710230410, "badge": "B-0101", "door": "lobby", "outcome": "DENIED"},
    {"ts": 1710230420, "badge": "B-0101", "door": "lobby", "outcome": "DENIED"},
    {"ts": 1710230430, "badge": "B-0101", "door": "lobby", "outcome": "DENIED"},
    {"ts": 1710230440, "badge": "B-0101", "door": "lobby", "outcome": "DENIED"},
]
far_pairs = set()
solve(entries, far_pairs)  ->  ["B-0101"]

entries2 = [
    {"ts": 1710288001, "badge": "B-0202", "door": "restricted-vault", "outcome": "GRANTED"},
]
# ts 1710288001 = 2024-03-13 02:00:01 UTC  (after-hours)
far_pairs2 = set()
solve(entries2, far_pairs2)  ->  ["B-0202"]
```

- [ ] Work the first example by hand and get the same answer

### Rules to respect
- [ ] Deterministic, offline, standard library only.
- [ ] Treat all timestamps as UTC when checking hour for after-hours (hour in {22,23,0,1,2,3,4,5}).
- [ ] Must pass all cases in the tests (visible + hidden).
- [ ] Write the steps your function will take, in plain words
- [ ] List two edge cases (an empty input, a tricky value)

<details><summary>Reference: the exact contract</summary>

Any implementation that satisfies this interface can be swapped in (reference, student, or
composite) and must pass the tests unchanged.

#### Entry point
```python
def solve(entries: list[dict], far_pairs: set[frozenset]) -> list[str]
```

#### Input (`entries`)
`list[dict]` where each dict has:
- `"ts"`: `int`, Unix timestamp in seconds (UTC)
- `"badge"`: `str`, badge identifier string
- `"door"`: `str`, door/reader name string
- `"outcome"`: `str`, either `"GRANTED"` or `"DENIED"`

#### Input (`far_pairs`)
`set[frozenset]`, each frozenset contains exactly two door name strings that are geographically
far apart (i.e., the same badge appearing at both within 600 seconds is a location inconsistency).

#### Output (`result`)
A **sorted, de-duplicated** `list[str]` of badge IDs that triggered one or more alert rules.
Empty list if no alerts.

#### Alert rules
- **Rule 0, Repeated denial burst:** count DENIED events per badge in any 60-second sliding
  window. Flag the badge if count >= 5.
- **Rule 1, After-hours restricted access:** flag a badge if it has any GRANTED entry at a door
  whose name starts with `"restricted-"` and whose timestamp UTC hour is in {22,23,0,1,2,3,4,5}.
- **Rule 2, Location inconsistency:** for each badge, scan GRANTED entries in timestamp order.
  If two consecutive GRANTED entries at different doors occur within 600 seconds AND that door pair
  is in `far_pairs`, flag the badge.

#### Guarantees / rules
- Pure function, deterministic, no I/O, no network. Standard library only.
- Entries may be given in any order; sort by `"ts"` internally as needed.
- Output always sorted ascending and de-duplicated.

#### Acceptance
Your code passes when every test below shows a checkmark (some are hidden until you run them).

</details>

## 3. Build it

This is the model solution.

- [ ] `solve` handles the worked examples
- [ ] `solve` handles your edge cases

In [ ]:
from typing import List, Set, FrozenSet, Dict
import datetime


def solve(entries: List[Dict], far_pairs: Set[FrozenSet]) -> List[str]:
    flagged = set()

    by_badge: Dict[str, List[Dict]] = {}
    for e in entries:
        by_badge.setdefault(e["badge"], []).append(e)

    after_hours = {22, 23, 0, 1, 2, 3, 4, 5}

    for badge, evts in by_badge.items():
        evts_sorted = sorted(evts, key=lambda x: x["ts"])

        denials = [e["ts"] for e in evts_sorted if e["outcome"] == "DENIED"]
        for i in range(len(denials)):
            window_end = denials[i] + 60
            count = sum(1 for ts in denials[i:] if ts <= window_end)
            if count >= 5:
                flagged.add(badge)
                break

        for e in evts_sorted:
            if e["outcome"] == "GRANTED" and e["door"].startswith("restricted-"):
                hour = datetime.datetime.utcfromtimestamp(e["ts"]).hour
                if hour in after_hours:
                    flagged.add(badge)
                    break

        granted = [e for e in evts_sorted if e["outcome"] == "GRANTED"]
        for i in range(len(granted) - 1):
            a, b = granted[i], granted[i + 1]
            if a["door"] != b["door"] and (b["ts"] - a["ts"]) <= 600:
                pair = frozenset({a["door"], b["door"]})
                if pair in far_pairs:
                    flagged.add(badge)
                    break

    return sorted(flagged)


## 4. Test it

The tests are the acceptance check, like a change review before a fix goes live.

- [ ] Run the tests
- [ ] Every test says PASS (if not, read the message, go back to step 3)

In [ ]:
import __main__ as impl

BASE = 1710316800
AH = 1710370800


def _e(ts, badge, door, outcome):
    return {"ts": ts, "badge": badge, "door": door, "outcome": outcome}


def test_rule0_burst_exactly_five():
    entries = [_e(BASE + i * 10, "B-0001", "lobby", "DENIED") for i in range(5)]
    assert impl.solve(entries, set()) == ["B-0001"]


def test_rule0_four_denials_no_flag():
    entries = [_e(BASE + i * 10, "B-0002", "lobby", "DENIED") for i in range(4)]
    assert impl.solve(entries, set()) == []


def test_rule1_after_hours_restricted():
    entries = [_e(AH, "B-0003", "restricted-server-room", "GRANTED")]
    assert impl.solve(entries, set()) == ["B-0003"]


def test_rule1_after_hours_non_restricted_no_flag():
    entries = [_e(AH, "B-0004", "lobby", "GRANTED")]
    assert impl.solve(entries, set()) == []


def test_rule2_location_inconsistency():
    fp = {frozenset({"door-A", "door-B"})}
    entries = [
        _e(BASE, "B-0005", "door-A", "GRANTED"),
        _e(BASE + 300, "B-0005", "door-B", "GRANTED"),
    ]
    assert impl.solve(entries, fp) == ["B-0005"]


def test_rule2_no_far_pair_no_flag():
    entries = [
        _e(BASE, "B-0006", "door-A", "GRANTED"),
        _e(BASE + 300, "B-0006", "door-C", "GRANTED"),
    ]
    fp = {frozenset({"door-A", "door-B"})}
    assert impl.solve(entries, fp) == []


def test_empty_entries():
    assert impl.solve([], set()) == []


def test_multiple_badges_some_flagged():
    entries = (
        [_e(BASE + i * 10, "B-0007", "lobby", "DENIED") for i in range(6)]
        + [_e(BASE + 100, "B-0008", "lobby", "DENIED")]
    )
    assert impl.solve(entries, set()) == ["B-0007"]


def test_rule0_spread_across_61s_no_flag():
    entries = [_e(BASE + i * 16, "B-0009", "lobby", "DENIED") for i in range(5)]
    assert impl.solve(entries, set()) == []


def test_rule1_boundary_hour_22():
    ts_22 = 1710370800 - 3600
    entries = [_e(ts_22, "B-0010", "restricted-vault", "GRANTED")]
    assert impl.solve(entries, set()) == ["B-0010"]


def test_rule2_gap_exactly_600s():
    fp = {frozenset({"site-X", "site-Y"})}
    entries = [
        _e(BASE, "B-0011", "site-X", "GRANTED"),
        _e(BASE + 600, "B-0011", "site-Y", "GRANTED"),
    ]
    assert impl.solve(entries, fp) == ["B-0011"]


def test_output_sorted_multiple_flags():
    entries = (
        [_e(BASE + i * 10, "B-0013", "lobby", "DENIED") for i in range(5)]
        + [_e(BASE + i * 10, "B-0012", "lobby", "DENIED") for i in range(5)]
    )
    result = impl.solve(entries, set())
    assert result == sorted(result)
    assert "B-0012" in result and "B-0013" in result


# ---- run every test above ------------------------------------------------
_results = []
for _name, _fn in list(globals().items()):
    if _name.startswith("test_") and callable(_fn):
        try:
            _fn()
            _results.append((_name, None))
        except Exception as _e:
            _results.append((_name, _e))
for _name, _e in _results:
    print(("PASS  " if _e is None else "FAIL  ") + _name + ("" if _e is None else f": {_e!r}"))
print(f"{sum(e is None for _, e in _results)}/{len(_results)} tests passed")


## 5. Try it on the evidence

Run your finished tool on the examples from the brief, then change one value and predict the answer before you run it again.

- [ ] Every example says OK
- [ ] Your own changed example gives the answer you predicted

In [ ]:
checks = [
    (lambda: solve([], set()),
     []),
]

for n, (run, want) in enumerate(checks, 1):
    try:
        got = run()
    except Exception as e:
        got = "error: " + repr(e)
    print("example", n, "OK  " if got == want else "DIFF", "got:", got, " expected:", want)


## 6. Wrap up

- [ ] Explain in one sentence why your rule catches the problem
- [ ] Name one way an attacker could slip past it (a false negative)
- [ ] Name one harmless input it might wrongly flag (a false positive)
- [ ] Say where this check would run in a real organisation, and who would act on it